In [ ]:
!pip install xgboost lightgbm pyngrok

In [18]:
df = pd.read_csv("car_price_prediction.csv")
df.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

In [23]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,19237.0,4.557654e+07,936591.422799,20746880.0,45698374.0,45772308.0,45802036.0,45816654.0
Price,19237.0,1.855593e+04,190581.269684,1.0,5331.0,13172.0,22075.0,26307500.0
Prod. year,19237.0,2.010913e+03,5.668673,1939.0,2009.0,2012.0,2015.0,2020.0
Cylinders,19237.0,4.582991e+00,1.199933,1.0,4.0,4.0,4.0,16.0
Airbags,19237.0,6.582627e+00,4.320168,0.0,4.0,6.0,12.0,16.0


In [35]:
df.Model.value_counts()

,count
Model,
Prius,1083
Sonata,1079
Camry,938
Elantra,922
E 350,542
...,...
530 i,1
E 500 AVG,1
Vito Extralong,1


In [28]:
df.isnull().sum()

,0
ID,0
Price,0
Levy,0
Manufacturer,0
Model,0
Prod. year,0
Category,0
Leather interior,0
Fuel type,0
Engine volume,0


In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn & Modeling
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ==========================================
# PHASE 1: Exploratory Data Analysis (EDA) & Cleaning
# ==========================================
def load_and_clean_data(filepath):
    print("Loading data...")
    df = pd.read_csv(filepath)

    # Drop irrelevant ID column
    if 'ID' in df.columns:
        df = df.drop('ID', axis=1)

    print("Initial Data Cleaning...")
    # 1. Clean Levy (Replace '-' with NaN, convert to float)
    df['Levy'] = df['Levy'].replace('-', np.nan).astype(float)

    # 2. Clean Mileage (Remove ' km' and convert to float)
    df['Mileage'] = df['Mileage'].astype(str).str.replace(' km', '').astype(float)

    # 3. Clean Engine volume (Extract numeric, create a Turbo indicator)
    df['Turbo'] = df['Engine volume'].astype(str).str.contains('Turbo').astype(int)
    df['Engine volume'] = df['Engine volume'].astype(str).str.replace(' Turbo', '').astype(float)

    # 4. Clean Doors (Standardize weird date-like strings)
    door_mapping = {'04-May': '4-5', '02-Mar': '2-3', '>5': '>5'}
    df['Doors'] = df['Doors'].map(door_mapping)

    # Remove extreme outliers in Price (e.g., typos in price, $1 cars, etc.)
    df = df[(df['Price'] > 500) & (df['Price'] < 500000)]

    return df

def perform_eda(df):
    print("Performing EDA...")
    # 1. Correlation Matrix Heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Correlation Matrix")
    plt.savefig('correlation_matrix.png')
    plt.close()

    # 2. Target Variable Distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(df['Price'], bins=50, kde=True)
    plt.title("Price Distribution")
    plt.savefig('price_distribution.png')
    plt.close()

# ==========================================
# PHASE 2: Advanced Preprocessing & Engineering
# ==========================================
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

def build_preprocessing_pipeline(num_features, cat_features_low_card, cat_features_high_card):
    # Numeric Pipeline: Impute -> RobustScaler (handles remaining outliers)
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

    # Low Cardinality Categorical Pipeline: Impute -> OneHotEncode
    cat_low_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # High Cardinality Categorical Pipeline: Impute -> OneHotEncode -> PCA
    # PCA reduces the massive dimensionality of Models/Manufacturers
    cat_high_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ('pca', PCA(n_components=50, random_state=42))
    ])

    # Combine via ColumnTransformer
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_features),
        ('cat_low', cat_low_pipeline, cat_features_low_card),
        ('cat_high', cat_high_pipeline, cat_features_high_card)
    ])

    return preprocessor

# ==========================================
# PHASE 3 & 4: Model Training, Eval & Comparison
# ==========================================
def evaluate_model(y_true, y_pred, n, p):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
    return rmse, mae, r2, adj_r2

def main():
    # 1. Load Data
    df = load_and_clean_data("car_price_prediction.csv")

    # IQR Outlier mitigation on target variable to stabilize training
    df = remove_outliers_iqr(df, 'Price')

    perform_eda(df)

    # Feature Segregation
    target = 'Price'
    X = df.drop(target, axis=1)
    y = df[target]

    num_features = ['Levy', 'Prod. year', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags', 'Turbo']
    cat_low_card = ['Category', 'Leather interior', 'Fuel type', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel']
    cat_high_card = ['Manufacturer', 'Model', 'Color']

    # 2. Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 3. Build Preprocessor
    preprocessor = build_preprocessing_pipeline(num_features, cat_low_card, cat_high_card)

    # 4. Define Models
    models = {
        'RandomForest (Baseline)': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
        'LightGBM': LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
    }

    results = []
    best_model = None
    best_r2 = -float('inf')
    best_pipeline = None

    n = len(X_test)
    p = X_train.shape[1]

    print("Training Models...")
    for name, model in models.items():
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])

        # Train
        pipeline.fit(X_train, y_train)

        # Predict
        y_pred = pipeline.predict(X_test)

        # Evaluate
        rmse, mae, r2, adj_r2 = evaluate_model(y_test, y_pred, n, p)
        results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R-Squared": r2, "Adj R-Squared": adj_r2})

        if r2 > best_r2:
            best_r2 = r2
            best_model = name
            best_pipeline = pipeline

    # 5. Display Comparison Table
    results_df = pd.DataFrame(results).sort_values(by="RMSE")
    print("\n--- Model Evaluation Comparison ---")
    print(results_df.to_string(index=False))

    print(f"\n🏆 Best Model: {best_model} with R2: {best_r2:.4f}")

    # 6. Save the Winning Model Pipeline
    joblib.dump(best_pipeline, 'car_price_model.pkl')
    print("Winning pipeline saved to 'car_price_model.pkl'")

if __name__ == "__main__":
    main()

Loading data...
Initial Data Cleaning...
Performing EDA...
Training Models...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13419
[LightGBM] [Info] Number of data points in the train set: 13208, number of used features: 87
[LightGBM] [Info] Start training from score 15412.575333


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



--- Model Evaluation Comparison ---
                  Model        RMSE         MAE  R-Squared  Adj R-Squared
RandomForest (Baseline) 5251.572265 3205.071660   0.771493       0.770310
               LightGBM 5339.243951 3606.759800   0.763800       0.762577
                XGBoost 5448.153082 3753.505859   0.754065       0.752793

🏆 Best Model: RandomForest (Baseline) with R2: 0.7715
Winning pipeline saved to 'car_price_model.pkl'


In [37]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

@st.cache_resource
def load_model():
    return joblib.load('car_price_model.pkl')

pipeline = load_model()

st.title("🚗 AI Car Price Prediction Engine")

# Unique categories extracted from the dataset (for UI population)
manufacturers = ['LEXUS', 'CHEVROLET', 'HONDA', 'FORD', 'HYUNDAI', 'TOYOTA', 'MERCEDES-BENZ', 'BMW', 'VOLKSWAGEN', 'AUDI'] # Truncated for example
models = ['RX 450', 'Equinox', 'FIT', 'Escape', 'Santa FE', 'Prius', 'Camry', 'E 350', 'X5'] # Truncated for example
colors = ['Silver', 'Black', 'White', 'Grey', 'Blue', 'Red']
categories = ['Jeep', 'Hatchback', 'Sedan', 'Microbus', 'Goods wagon', 'Universal', 'Coupe', 'Minivan', 'Cabriolet', 'Limousine', 'Pickup']

st.set_page_config(page_title="Car Price Predictor", layout="wide")

st.markdown("Enter the specifications of the vehicle below to get a real-time price estimation.")

# Layout: Split into Categorical and Numerical Sections
col1, col2 = st.columns(2)

with col1:
    st.subheader("Categorical Features")
    st.caption("Use the searchable multi-selects for high cardinality fields.")

    # High Cardinality -> Searchable Multiselect (restricted to 1 to mimic exact selection)
    manufacturer = st.multiselect("Manufacturer", manufacturers, max_selections=1, default=["TOYOTA"])
    car_model = st.multiselect("Model", models, max_selections=1, default=["Camry"])
    color = st.multiselect("Color", colors, max_selections=1, default=["Black"])

    # Low Cardinality
    category = st.selectbox("Category", categories)
    fuel_type = st.selectbox("Fuel Type", ['Petrol', 'Diesel', 'Hybrid', 'LPG', 'CNG', 'Plug-in Hybrid', 'Hydrogen'])
    gear_box = st.selectbox("Gear Box Type", ['Automatic', 'Tiptronic', 'Manual', 'Variator'])
    drive_wheels = st.selectbox("Drive Wheels", ['4x4', 'Front', 'Rear'])
    doors = st.selectbox("Doors", ['4-5', '2-3', '>5'])
    wheel = st.selectbox("Wheel", ['Left wheel', 'Right-hand drive'])
    leather = st.checkbox("Leather Interior")

with col2:
    st.subheader("Numerical Features")
    st.caption("Enter the exact digits for the vehicle's specs.")

    prod_year = st.number_input("Production Year", min_value=1900, max_value=2026, value=2015, step=1)
    mileage = st.number_input("Mileage (km)", min_value=0, value=100000, step=1000)
    engine_vol = st.number_input("Engine Volume (L)", min_value=0.0, value=2.0, step=0.1)
    turbo = st.checkbox("Has Turbo?")
    cylinders = st.number_input("Cylinders", min_value=1, max_value=16, value=4, step=1)
    airbags = st.number_input("Airbags", min_value=0, max_value=20, value=4, step=1)
    levy = st.number_input("Levy (Tax/Import Fee)", min_value=0.0, value=1000.0, step=50.0)

st.markdown("---")

# Predict Button
if st.button("🔮 Predict Price", type="primary", use_container_width=True):
    # Construct input payload matching training data schema
    input_data = pd.DataFrame({
        'Manufacturer': [manufacturer[0] if manufacturer else 'TOYOTA'],
        'Model': [car_model[0] if car_model else 'Camry'],
        'Color': [color[0] if color else 'Black'],
        'Category': [category],
        'Fuel type': [fuel_type],
        'Gear box type': [gear_box],
        'Drive wheels': [drive_wheels],
        'Doors': [doors],
        'Wheel': [wheel],
        'Leather interior': ['Yes' if leather else 'No'],
        'Prod. year': [prod_year],
        'Mileage': [mileage],
        'Engine volume': [engine_vol],
        'Turbo': [1 if turbo else 0],
        'Cylinders': [cylinders],
        'Airbags': [airbags],
        'Levy': [levy]
    })

    with st.spinner("Processing through ML Pipeline..."):
        try:
            # Inference through the exact same Pipeline
            prediction = pipeline.predict(input_data)[0]

            st.success("Prediction Successful!")
            st.markdown(f"### 💰 Estimated Market Price: **${prediction:,.2f}**")

        except Exception as e:
            st.error(f"An error occurred during prediction: {str(e)}")

Overwriting app.py


In [ ]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.7 MB/s eta 0:00:00


In [ ]:
# 1. تثبيت cloudflared
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 2. تشغيل Streamlit و Cloudflare Tunnel معاً
import subprocess
import time

# تشغيل Streamlit في الخلفية
subprocess.Popen(["streamlit", "run", "app.py"])

# انتظار ثانيتين ليتأكد من تشغيل Streamlit
time.sleep(2)

# تشغيل Cloudflare ورؤية رابط الموقع الجاهز
!./cloudflared tunnel --url http://localhost:8501

2026-08-11T23:35:43Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-11T23:35:43Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-11T23:35:47Z INF +--------------------------------------------------------------------------------------------+
2026-08-11T23:35:47Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-11T23:35:47Z INF |  https://wednesday-tower-labeled-familiar.trycloudflar